In [ ]:
# OmniVoice's own lockfile, from PyTorch's cu128 index for torch; torchvision at
# the version that belongs to that torch so transformers can import beside it.
!pip -q install "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3
!pip -q install "transformers==5.3.0" "accelerate==1.13.0" "webdataset==1.0.2" "safetensors==0.7.0" "tokenizers==0.22.2" "huggingface-hub==1.7.2" "tensorboardX==2.6.4" "soundfile==0.13.1" "librosa==0.11.0" "pydub==0.25.1" 2>&1 | tail -3
!pip -q install --no-deps "git+https://github.com/k2-fsa/OmniVoice@08be0b4ccbac3e13e374e86fbfead4b4cac343e2" 2>&1 | tail -3
!pip -q install speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime 2>&1 | tail -3


In [ ]:
import hashlib
import time as _clock
NOTEBOOK_START = _clock.time()
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def captured(command):
    result = subprocess.run(command, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    print(captured([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])[1])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import torch

OFFICIAL = {'torch': '2.8.0+cu128', 'torchaudio': '2.8.0+cu128', 'transformers': '5.3.0', 'accelerate': '1.13.0', 'webdataset': '1.0.2', 'safetensors': '0.7.0', 'tokenizers': '0.22.2', 'huggingface-hub': '1.7.2', 'tensorboardX': '2.6.4', 'soundfile': '0.13.1', 'librosa': '0.11.0', 'pydub': '0.25.1', 'numpy': '2.4.3', 'triton': '3.4.0'}
PINNED = ('torch', 'torchaudio', 'transformers', 'accelerate', 'webdataset', 'safetensors', 'tokenizers', 'huggingface-hub')


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def matches(have, want):
    if have is None:
        return False
    return have == want or (have.split("+")[0] == want.split("+")[0] and "+" not in want)


stack = {name: {"official": want, "installed": installed(name),
                "matches": matches(installed(name), want), "pinned_here": name in PINNED}
         for name, want in OFFICIAL.items()}
direct = json.loads(importlib.metadata.distribution("omnivoice").read_text("direct_url.json") or "{}")
code, driver = captured(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"])
ENVIRONMENT_RECORD = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "bf16_supported_including_emulation": torch.cuda.is_bf16_supported(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "nvidia_smi": driver,
    "official_stack": stack,
    "omnivoice": {"version": installed("omnivoice"),
                  "commit": (direct.get("vcs_info") or {}).get("commit_id"), "url": direct.get("url")},
    "other_packages": {name: installed(name) for name in ("torchvision", "ipykernel", "papermill", "scipy")},
}
Path("/kaggle/working/environment.json").write_text(json.dumps(ENVIRONMENT_RECORD, indent=2))
for name, row in stack.items():
    mark = "ok" if row["matches"] else ("DIFFERS" if row["pinned_here"] else "differs - image's version kept")
    print(f"{name:16} official {row['official']:12} installed {str(row['installed']):16} {mark}")
print("omnivoice", ENVIRONMENT_RECORD["omnivoice"])
print("cuda", torch.version.cuda, "| cudnn", torch.backends.cudnn.version(), "|", driver)
for name in PINNED:
    assert stack[name]["matches"], f"{name} is not the official {OFFICIAL[name]}: {stack[name]['installed']}"
assert ENVIRONMENT_RECORD["omnivoice"]["commit"] == "08be0b4ccbac3e13e374e86fbfead4b4cac343e2", ENVIRONMENT_RECORD["omnivoice"]
assert torch.cuda.device_count() >= 2, "the official recipe trains on two GPUs - use T4 x2"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}, capability {torch.cuda.get_device_capability(i)}, "
          f"{free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk:", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GiB free under /tmp")

INPUT = Path("/kaggle/input")
found = sorted(p for p in INPUT.rglob("repo/scripts/omnivoice_adapt.py") if "persian-voice-eval" in p.parts)
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo with scripts/omnivoice_adapt.py under /kaggle/input")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
# Kaggle unpacks the .gz when it builds the dataset; either form is checked
# against the hash the split builder recorded for its decompressed content.
MANIFESTS = REPO / "eval/manifests"
SPLITS = next((p for p in (MANIFESTS / "adaptation_splits_v1.jsonl.gz", MANIFESTS / "adaptation_splits_v1.jsonl")
               if p.is_file()), None)
assert SPLITS is not None, "the dataset predates the adaptation splits"
import gzip  # noqa: E402

content = gzip.open(SPLITS).read() if SPLITS.suffix == ".gz" else SPLITS.read_bytes()
expected = json.loads((MANIFESTS / "adaptation_splits_v1.summary.json").read_text())["records_sha256"]
assert hashlib.sha256(content).hexdigest() == expected, "the split file is not the one the builder wrote"
print("splits:", SPLITS.name, "sha256 matches the summary")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-omnivoice-test-v1"
sys.path.insert(0, str(REPO / "scripts"))
from kernel_gates import Gates, report_group, run_bounded  # noqa: E402

GATES = Gates()
RUNS = {}
LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
WORK = Path("/tmp/adapt")
WORK.mkdir(parents=True, exist_ok=True)
RECORDS = Path("/kaggle/working/records")
RECORDS.mkdir(exist_ok=True)


In [ ]:
from runner_check import check, diagnostics  # noqa: E402

ok, reason, run = check(run_bounded, Path("/kaggle/working/runner_check"), REPO / "scripts/moss15_worker.py")
RUNS["runner_check"] = run.record()
print(run.summary())
seen, line = diagnostics(report_group)
print("descriptor report:", "works" if seen else "DOES NOT WORK", "-", line)
GATES.record("runner", ok, reason)


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


from huggingface_hub import snapshot_download

os.environ["VOICECLONE_OMNIVOICE_PYTHON"] = sys.executable
provenance = found[0].parents[2] / "PROVENANCE.md"
os.environ["VOICECLONE_CODE_COMMIT"] = (provenance.read_text().split("commit ")[-1].strip(" .\n")
                                        if provenance.is_file() else "the dataset's PROVENANCE.md was not found")
PARQUET = {"neyshekar": Path("/tmp/neyshekar-test"), "cv26": Path("/tmp/cv26")}
TEXTS = REPO / "eval/manifests/test_texts_v1.json"
BASE = None
# A dataset staged before this code existed would fail only after the downloads.
NEEDED = ["scripts/prepare_test_corpus.py", "scripts/read_sealed_test.py", "scripts/build_adaptation_splits.py",
          "eval/manifests/test_texts_v1.json"]
missing = [name for name in NEEDED if not (REPO / name).is_file()]
if "--sentences-file" not in (REPO / "scripts/benchmark.py").read_text():
    missing.append("benchmark.py --sentences-file")
GATES.record("code", not missing, f"missing from the dataset: {missing}" if missing else "the test code is present")
if not GATES.skip("download", ("runner", "code")):
    try:
        start = time.time()
        BASE = Path(snapshot_download("k2-fsa/OmniVoice", revision="c5fdb5ccb189668d56333f77ba2629f4cd7535f4", local_dir="/tmp/omnivoice-base"))
        # Neyshekar's test split is the primary test set; only its test shards are fetched.
        snapshot_download("shekar-ai/neyshekar", repo_type="dataset", revision="30696ada746a66bb3047c984744b5ecfd60fe4e9",
                          allow_patterns=["data/test-*"], local_dir=str(PARQUET["neyshekar"]))
        snapshot_download("pymmdrza/Common-Voice-Speech-26.0-Persian-Clean", repo_type="dataset", revision="142e9bb40b56d9491faa0ec0987ee8d408bd9c05",
                          allow_patterns=["data/*"], local_dir=str(PARQUET["cv26"]))
        shards = sorted(p.name for p in PARQUET["neyshekar"].rglob("*.parquet"))
        assert shards and all(name.startswith("test-") for name in shards), shards
        GATES.record("download", True, f"{(time.time() - start) / 60:.1f} min; neyshekar {shards}")
    except Exception as exc:  # noqa: BLE001 - the gate records it
        GATES.record("download", False, f"{type(exc).__name__}: {exc}")


In [ ]:
SETS = {'ney': ('neyshekar', 'primary', 28), 'cv': ('cv26', 'secondary', 30)}
CORPORA = {}
if not GATES.skip("corpora", ("download",)):
    ok = True
    for name, (source, role, planned) in SETS.items():
        out = Path(f"/tmp/test-{name}")
        run = run_bounded([sys.executable, str(REPO / "scripts/prepare_test_corpus.py"), "--splits", str(SPLITS),
                           "--source", source, "--parquet-dir", str(PARQUET[source]), "--out", str(out)],
                          timeout=2 * 3600, log=LOGS / f"corpus_{name}.log", monitor_every=600)
        RUNS[f"corpus_{name}"] = run.record()
        print(run.summary())
        if not run.ok or not (out / "roster.json").is_file():
            ok = False
            continue
        roster = json.loads((out / "roster.json").read_text())
        CORPORA[name] = {"dir": out, "roster": roster, "speakers": len(roster["speakers"]), "planned": planned}
        # Fewer than planned is reported, never topped up: no speaker is added after looking.
        print(name, len(roster["speakers"]), "speakers of", planned, "| replaced", roster["replaced"],
              "| left out", roster["left_out"])
        ok = ok and len(roster["speakers"]) >= 5
    GATES.record("corpora", ok, json.dumps({name: {k: v for k, v in c.items() if k != "dir"}
                                             for name, c in CORPORA.items()}, default=str))


In [ ]:
CANDIDATE = {}
EXPORT = None
if not GATES.skip("candidate", ("corpora",)):
    choices = sorted(INPUT.rglob("choice.json"))
    choice = json.loads(choices[0].read_text()) if len(choices) == 1 else {}
    best = choice.get("candidate")
    if best:
        matches = sorted(p for p in INPUT.rglob(f"export-{best['step']}-float16") if p.is_dir())
        EXPORT = matches[0] if len(matches) == 1 else None
    weights = sorted(EXPORT.glob("*.safetensors")) if EXPORT else []
    CANDIDATE = {"choice_files": [str(p) for p in choices], "rule": choice.get("rule"), "candidate": best,
                 "export": str(EXPORT) if EXPORT else None,
                 "weights_sha256": {p.name: sha256(p) for p in weights}}
    GATES.record("candidate", bool(best) and EXPORT is not None and bool(weights), json.dumps(CANDIDATE))


In [ ]:
MODEL_KEY = "VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"
SEEDS = [42, 43]
LABELS = {}


def remaining_hours():
    return 11.25 - (time.time() - NOTEBOOK_START) / 3600


def bench(label, model, corpus_dir, speakers, *extra):
    timeout = min(3.0, remaining_hours() - 0.5) * 3600
    if timeout < 1800:
        print(f"{label}: {remaining_hours():.2f} h left in the session - not started")
        return False, {}
    saved = os.environ.get(MODEL_KEY)
    os.environ[MODEL_KEY] = str(model)
    try:
        run = run_bounded([sys.executable, "scripts/benchmark.py", "--label", label, "--engine", "omnivoice",
                           "--corpus-dir", str(corpus_dir), "--sentences-file", str(TEXTS),
                           "--seeds", *map(str, SEEDS), "--device", "cuda", *extra],
                          cwd=REPO, timeout=timeout, log=LOGS / f"{label}.log", monitor_every=600)
    finally:
        if saved is None:
            os.environ.pop(MODEL_KEY, None)
        else:
            os.environ[MODEL_KEY] = saved
    RUNS[label] = run.record()
    print(run.summary())
    results = REPO / "eval/results" / f"{label}.json"
    summary = json.loads(results.read_text())["summary"] if results.is_file() else {}
    expected = speakers * 12 * len(SEEDS)
    return run.ok and summary.get("failures", 1) == 0 and summary.get("renders", 0) == expected, summary


if not GATES.skip("renders", ("candidate",)):
    ok = True
    facts = {}
    for name, (source, role, planned) in SETS.items():
        corpus = CORPORA[name]
        published = f"omnivoice-published-test-{name}"
        candidate = f"omnivoice-fa-{CANDIDATE['candidate']['step']}-test-{name}"
        LABELS[name] = {"published": published, "candidate": candidate, "role": role}
        # The recogniser's floor is measured on this set's real clips, by the first run.
        passed_published, summary_published = bench(published, BASE, corpus["dir"], corpus["speakers"], "--asr-floor")
        passed_candidate, summary_candidate = bench(candidate, EXPORT, corpus["dir"], corpus["speakers"])
        facts[name] = {"published": summary_published.get("held_out_mean"),
                       "candidate": summary_candidate.get("held_out_mean"),
                       "complete": [passed_published, passed_candidate]}
        ok = ok and passed_published and passed_candidate
    GATES.record("renders", ok, json.dumps(facts))


In [ ]:
import compare_runs  # noqa: E402
import read_sealed_test  # noqa: E402

READING = {}
if not GATES.skip("reading", ("candidate",)):
    for name, labels in LABELS.items():
        try:
            READING[name] = read_sealed_test.reading(
                labels["candidate"], compare_runs.load(labels["candidate"]),
                labels["published"], compare_runs.load(labels["published"]),
                labels["role"], read_sealed_test.categories_of(TEXTS))
        except SystemExit as exc:
            READING[name] = {"error": str(exc)}
        print(name, json.dumps(READING[name].get("verdict", READING[name]), ensure_ascii=False))
    GATES.record("reading", bool(READING) and all("verdict" in r for r in READING.values()),
                 json.dumps({name: r.get("verdict") for name, r in READING.items()}))


In [ ]:
destination = Path("/kaggle/working/test")
destination.mkdir(exist_ok=True)
for path in (REPO / "eval/results").glob("*.json"):
    shutil.copy2(path, destination / path.name)
(destination / "reading.json").write_text(json.dumps(READING, indent=2, ensure_ascii=False))
(destination / "candidate.json").write_text(json.dumps(CANDIDATE, indent=2))
(destination / "gates.json").write_text(json.dumps(GATES.summary(), indent=2))
(destination / "runs.json").write_text(json.dumps(RUNS, indent=2))
shutil.copy2("/kaggle/working/environment.json", destination / "environment.json")
for name, corpus in CORPORA.items():
    # The real clips stay with the renders: a listening test needs both.
    shutil.copytree(corpus["dir"], Path("/kaggle/working/corpora") / name, dirs_exist_ok=True)
renders = Path("/kaggle/working/renders")
renders.mkdir(exist_ok=True)
for labels in LABELS.values():
    for label in (labels["published"], labels["candidate"]):
        folder = REPO / "outputs/benchmark" / label
        for path in folder.glob("*.wav") if folder.is_dir() else []:
            shutil.copy2(path, renders / f"{label}__{path.name}")
print("gates:", json.dumps(GATES.summary(), indent=1))
